# PHASE 2: DATA CLEANING & PREPROCESSING PIPELINE
**Học phần:** DTA301 – Data Analytics
**Người thực hiện:** Lê Đỗ Nhật Anh (SE180055)
**Mục tiêu:**
- Tải và nạp dữ liệu telemetry UAV & giám sát tấm pin.
- Xử lý missing values (1 nan ở speed_diff, 9 nan ở imu_var).
- Ép kiểu dữ liệu (Type casting) và ràng buộc vật lý.
- Xuất dữ liệu sạch sang thư mục `data/processed/`.

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import gdown

# 1. Cố định tính tái lập (Reproducibility) theo chuẩn môn học DTA301
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 2. Khởi tạo đường dẫn thư mục lưu trữ
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# 3. Tải dữ liệu thô nếu chưa tồn tại
feat_path = "../data/raw/engineered_features_labels.csv"
if not os.path.exists(feat_path):
    gdown.download(id='16Fhb79Md7xeRd1UnvLZX1O_5ZcBXbUCJ', output=feat_path, quiet=False)

# 4. Đọc dữ liệu và kiểm tra cấu trúc ban đầu
df_feat = pd.read_csv(feat_path)
print("Kích thước dataset:", df_feat.shape)
print("\nKiểm tra missing values ban đầu:")
print(df_feat.isnull().sum())

Downloading...
From: https://drive.google.com/uc?id=16Fhb79Md7xeRd1UnvLZX1O_5ZcBXbUCJ
To: /content/data/raw/engineered_features_labels.csv
100%|██████████| 516k/516k [00:00<00:00, 113MB/s]

Kích thước dataset: (5000, 9)

Kiểm tra missing values ban đầu:
speed               0
speed_diff          1
imu_var             9
battery_drop        0
unstable_battery    0
wind_spike          0
unstable_wind       0
gps_anomaly         0
unstable_flight     0
dtype: int64


In [ ]:
# ==============================================================================
# LÀM SẠCH VÀ CHUẨN BỊ DỮ LIỆU SURVEILDRONE-NET23 (PHỤC VỤ RQ2)
# ==============================================================================
surveil_raw_path = "../data/raw/SurveilDrone-Net23.csv"

if os.path.exists(surveil_raw_path):
    print("--- ĐANG NẠP VÀ LÀM SẠCH SURVEILDRONE-NET23 ---")
    df_surveil = pd.read_csv(surveil_raw_path)
    print(f"Kích thước ban đầu: {df_surveil.shape}")
    
    # 1. Loại bỏ các dòng trùng lặp (nếu có)
    df_surveil = df_surveil.drop_duplicates()
    
    # 2. Ràng buộc vật lý (Operational Boundary Validation)
    if 'battery_level_pct' in df_surveil.columns:
        df_surveil = df_surveil[(df_surveil['battery_level_pct'] >= 0) & (df_surveil['battery_level_pct'] <= 100)]
        
    if 'wind_speed_mps' in df_surveil.columns:
        df_surveil = df_surveil[(df_surveil['wind_speed_mps'] >= 0) & (df_surveil['wind_speed_mps'] <= 35)]
        
    # 3. Chuẩn hóa Datetime
    if 'timestamp' in df_surveil.columns:
        df_surveil['timestamp'] = pd.to_datetime(df_surveil['timestamp'])
        
    # 4. Xuất file sạch vào data/processed
    surveil_clean_path = "../data/processed/SurveilDrone_Net23_cleaned.csv"
    df_surveil.to_csv(surveil_clean_path, index=False)
    print(f"-> THÀNH CÔNG: Đã lưu {surveil_clean_path}")
    print(f"Kích thước sau khi làm sạch: {df_surveil.shape}")
else:
    print("Chưa tìm thấy file SurveilDrone-Net23.csv trong data/raw/.")

In [ ]:
# 1. Xử lý Missing Values theo đúng cam kết trong tài liệu:
# - speed_diff: 1 giá trị khuyết ở đầu -> nội suy tuyến tính (linear) hoặc bfill
df_feat['speed_diff'] = df_feat['speed_diff'].interpolate(method='linear').bfill()

# - imu_var: 9 giá trị khuyết ở đầu -> nội suy tuyến tính, nếu còn khuyết thì điền median
df_feat['imu_var'] = df_feat['imu_var'].interpolate(method='linear').fillna(df_feat['imu_var'].median())

# 2. Ép kiểu dữ liệu (Type Casting) cho các nhãn phân loại nhị phân
binary_cols = ['unstable_flight', 'gps_anomaly', 'unstable_battery', 'unstable_wind']
for col in binary_cols:
    if col in df_feat.columns:
        df_feat[col] = df_feat[col].astype(int)

# 3. Kiểm tra lại dữ liệu sau khi clean
print("--- KIỂM TRA LẠI SAU KHI LÀM SẠCH ---")
print("Missing values còn lại:\n", df_feat.isnull().sum())
print("\nKiểu dữ liệu các cột:\n", df_feat.dtypes)

# 4. Xuất file dữ liệu sạch vào data/processed/
processed_path = "../data/processed/engineered_features_cleaned.csv"
df_feat.to_csv(processed_path, index=False)
print(f"\n-> THÀNH CÔNG: Đã lưu dữ liệu sạch vào {processed_path}")

--- KIỂM TRA LẠI SAU KHI LÀM SẠCH ---
Missing values còn lại:
 speed               0
speed_diff          0
imu_var             0
battery_drop        0
unstable_battery    0
wind_spike          0
unstable_wind       0
gps_anomaly         0
unstable_flight     0
dtype: int64

Kiểu dữ liệu các cột:
 speed               float64
speed_diff          float64
imu_var             float64
battery_drop        float64
unstable_battery      int64
wind_spike          float64
unstable_wind         int64
gps_anomaly           int64
unstable_flight       int64
dtype: object

-> THÀNH CÔNG: Đã lưu dữ liệu sạch vào data/processed/engineered_features_cleaned.csv
